<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../5.%20building_apis/13.%20api_authentication_with_jwt.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 13: JWT Auth</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='15.%20error_handling_and_logging.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 15: Error Handling →</a>
</div>

# Chapter 14: File Uploads and Static Assets
## Handling Files Safely

> *"Users want to upload profile pictures, blog post images, and attachments. Handling files seems simple until you realize: malicious users can upload executable scripts, enormous files, or files with dangerous names. Safe file handling requires validation at every step."*

## 🎯 The Big Picture

File uploads are one of the most **mishandled** features in web development. Developers often write:

```python
file = request.files['photo']
file.save('/uploads/' + file.filename)  # Dangerous in multiple ways!
```

This innocent-looking code has at least **three serious vulnerabilities**:
1. `../../etc/passwd` as filename → **path traversal** attack
2. `malware.php` or `shell.py` → **executable upload**
3. A 10GB file → **disk space exhaustion / DoS**

In this chapter, you'll learn to handle files **correctly**: validate type, size, and content; sanitise filenames; generate safe unique names; and serve files securely.

**What we cover:**
- How multipart/form-data uploads work
- `secure_filename()` — the function you must never skip
- Validation checklist: extension, size, MIME type (magic bytes)
- Generating unique filenames with UUIDs
- Dated directory organisation for uploads
- Serving uploaded files safely with `send_from_directory()`
- Flask's static file system and `url_for('static', ...)`
- When to move from local storage to the cloud

## 🧠 Core Concepts: The "Why"

### How File Uploads Work — The Full Pipeline

The key to this section is sequence. If you can explain what happens first, what happens next, and where control moves after that, the moving parts here become much easier to reason about.

```json
[HTML Form]              [Browser]               [Flask Server]
<form enctype=           encodes file as          request.files['photo']
  multipart/form-data>   multipart/form-data      -> FileStorage object
<input type="file">      -> HTTP POST request     -> .save() to disk
```

The critical HTML attribute is **`enctype="multipart/form-data"`**. Without it, the browser sends only the filename, not the file content.
### The `FileStorage` Object

Werkzeug wraps each uploaded file in a `FileStorage` object:

| Attribute/Method | What it gives you |
|---|---|
| `file.filename` | Original filename from client (⚠️ **untrusted!**) |
| `file.content_type` | MIME type sent by browser (⚠️ **can be faked!**) |
| `file.stream` | Raw byte stream |
| `file.read()` | Read the file bytes |
| `file.save(path)` | Save to a path on disk |

### The Attacker's Toolkit

| Attack | Method | Consequence |
|---|---|---|
| **Path traversal** | Filename: `../../etc/passwd` | Read/overwrite arbitrary files |
| **Executable upload** | File: `shell.php` or `malware.py` | Remote code execution |
| **DoS via large files** | Upload a 10GB file | Disk full, server crash |
| **Filename collision** | Same filename as existing file | Data overwritten silently |
| **Extension spoofing** | `evil.php` renamed to `photo.jpg` | Bypass extension whitelist |

**Defence**: Validate everything. Trust nothing from the client.

## ⚡ Syntax & First Usage

### The HTML Form

In [ ]:
html_form = """
<form method="POST" action="/upload" enctype="multipart/form-data">
    <!-- enctype="multipart/form-data" is REQUIRED for file uploads -->
    <!-- Without it, only the filename is sent, not the content -->

    <label for="photo">Choose a photo:</label>
    <input type="file" name="photo" id="photo" accept=".jpg,.jpeg,.png,.gif">
    <!--  name="photo" matches request.files['photo'] -->
    <!--  accept=... is a browser hint only, NOT a security measure -->

    <button type="submit">Upload</button>
</form>
"""

print("Key HTML attributes:")
print("  enctype='multipart/form-data'  REQUIRED - enables file content upload")
print("  name='photo'                   matches request.files['photo']")
print("  accept='.jpg,.png'             browser filter only (not security!)")
print("  multiple                       allow multiple files at once")

### Basic (but Safe) File Upload Handler

In [ ]:
import os
from flask import Flask, request, jsonify
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = '/tmp/uploads'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

@app.route('/upload', methods=['POST'])
def upload():
    # 1. Check the file is present
    if 'photo' not in request.files:
        return jsonify(error='No file in request'), 400

    file = request.files['photo']

    # 2. An empty file input sends FileStorage with filename ''
    if file.filename == '':
        return jsonify(error='No file selected'), 400

    # 3. ALWAYS sanitise the filename
    filename = secure_filename(file.filename)

    # 4. Save to the upload folder
    save_path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
    file.save(save_path)

    return jsonify(message=f'Saved as: {filename}'), 200

print("Minimal upload handler")
print()
print("Steps:")
print("  1. Check 'photo' is in request.files")
print("  2. Check filename is not empty")
print("  3. secure_filename() to sanitise")
print("  4. file.save() to persist to disk")

### Why `secure_filename()` Is Non-Negotiable

In [ ]:
from werkzeug.utils import secure_filename

dangerous_filenames = [
    "../../etc/passwd",
    "/etc/cron.d/malicious",
    "../config.py",
    "photo with spaces.jpg",
    "shell.php.jpg",
    ".htaccess",
    "photo;rm -rf /.jpg",
    "",
]

print(f"{'Input':<35}  ->  {'Output'}")
print("-" * 60)
for name in dangerous_filenames:
    safe = secure_filename(name)
    print(f"{repr(name):<35}  ->  {repr(safe)}")

print()
print("secure_filename() removes:")
print("  - Path separators (/ and \\\\)")
print("  - Leading dots (hidden files)")
print("  - Special shell characters")
print("  - Non-ASCII characters")
print()
print("Always check for empty result:")
print("  filename = secure_filename(file.filename)")
print("  if not filename:")
print("      return jsonify(error='Invalid filename'), 400")

## 🔬 Deep Dive

### The Three-Layer Validation Stack

Never rely on a single check — layer your defences:

In [ ]:
import os, uuid
from flask import Flask, request, jsonify
from werkzeug.utils import secure_filename

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = '/tmp/uploads'
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16 MB hard limit

# LAYER 1: Extension Whitelist
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif', 'webp'}

def allowed_extension(filename):
    if '.' not in filename:
        return False
    ext = filename.rsplit('.', 1)[1].lower()
    return ext in ALLOWED_EXTENSIONS

# LAYER 2: MIME Type / Magic Bytes Check
def allowed_mime_type(file_stream):
    """
    Read the actual bytes of the file, not just the extension.
    Uses imghdr (stdlib) to check file magic bytes.
    For production: use python-magic for better coverage.
    """
    import imghdr
    file_stream.seek(0)
    img_type = imghdr.what(file_stream)
    file_stream.seek(0)
    return img_type in {'jpeg', 'png', 'gif', 'webp'}

# LAYER 3: Unique Filename Generation
def make_unique_filename(original_filename):
    _, ext = os.path.splitext(secure_filename(original_filename))
    return uuid.uuid4().hex + ext.lower()

# Full validated upload handler
@app.route('/upload', methods=['POST'])
def upload():
    if 'photo' not in request.files:
        return jsonify(error='No file provided'), 400
    file = request.files['photo']
    if not file or file.filename == '':
        return jsonify(error='No file selected'), 400

    # Layer 1: Extension
    if not allowed_extension(file.filename):
        return jsonify(error=f"Type not allowed. Use: {', '.join(ALLOWED_EXTENSIONS)}"), 415

    # Layer 2: MIME type (magic bytes — can't be faked by renaming)
    if not allowed_mime_type(file.stream):
        return jsonify(error='File content is not a valid image'), 415

    # Layer 3: Unique filename
    filename = make_unique_filename(file.filename)
    os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
    file.save(os.path.join(app.config['UPLOAD_FOLDER'], filename))
    return jsonify(filename=filename, url=f'/uploads/{filename}'), 201

print("Three-layer validation:")
print("  Layer 1: Extension whitelist  - fast, easy to bypass alone")
print("  Layer 2: MIME/magic bytes     - reads actual file content")
print("  Layer 3: UUID filename        - prevents collisions & overwriting")
print()
print("MAX_CONTENT_LENGTH = 16MB")
print("  Flask rejects oversized requests BEFORE the view runs.")
print("  No extra code needed in the handler.")

### Extension vs MIME Type — The Critical Difference

In [ ]:
print("Extension spoofing attack:")
print()
print("A malicious user has shell.php (a webshell)")
print("They rename it to: profile_photo.jpg")
print()
print("Extension-only check:")
print("  ext = 'jpg'")
print("  'jpg' in ALLOWED_EXTENSIONS  ->  True (PASSES!)")
print("  File saved as profile_photo.jpg")
print("  If server executes PHP... attacker has code execution")
print()
print("MIME type / magic bytes check:")
print("  File starts with: <?php echo shell_exec($_GET['cmd']); ?>")
print("  Magic bytes: 3C 3F 70 68 70 (that's '<' '?' 'p' 'h' 'p')")
print("  imghdr.what() returns: None (not a valid image)")
print("  -> 415 Unsupported Media Type (BLOCKED!)")
print()
print("Magic bytes for real image formats:")
magic = {
    'JPEG': 'FF D8 FF',
    'PNG':  '89 50 4E 47 0D 0A 1A 0A  (\\x89PNG\\r\\n\\x1a\\n)',
    'GIF':  '47 49 46 38  (GIF8)',
    'WebP': '52 49 46 46 ... 57 45 42 50  (RIFF...WEBP)',
}
for fmt, info in magic.items():
    print(f"  {fmt}: {info}")
print()
print("python-magic (libmagic) is more reliable than imghdr:")
print("  pip install python-magic")
print("  import magic")
print("  mime = magic.from_buffer(file.read(2048), mime=True)")
print("  # Returns 'image/jpeg', 'application/x-php', etc.")

### Generating Unique Filenames with UUIDs

In [ ]:
import uuid, os
from werkzeug.utils import secure_filename

def make_filename(original, strategy='uuid'):
    _, ext = os.path.splitext(secure_filename(original) if original else 'file.bin')
    ext = ext.lower()
    if strategy == 'uuid':
        return uuid.uuid4().hex + ext
    elif strategy == 'uuid_prefix':
        safe_name = os.path.splitext(secure_filename(original))[0][:20]
        return f"{safe_name}_{uuid.uuid4().hex[:8]}{ext}"
    elif strategy == 'timestamp':
        from datetime import datetime
        ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S_%f')
        return f"{ts}{ext}"

examples = ['profile.jpg', 'my photo.png', '../../etc/passwd', 'report.pdf']
print("Filename generation strategies:")
print()
for name in examples:
    print(f"Original: {name!r}")
    print(f"  uuid:         {make_filename(name, 'uuid')}")
    print(f"  uuid_prefix:  {make_filename(name, 'uuid_prefix')}")
    print(f"  timestamp:    {make_filename(name, 'timestamp')}")
    print()

print("UUID is safest: original filename info never leaks")
print("  User uploads 'confidential_salary_2024.xlsx' -> stored as 'a3f8b2c1....xlsx'")

### Date-Based Directory Organisation

Prevents single directories with millions of files (a filesystem performance problem):

In [ ]:
import os
from datetime import datetime

def get_upload_path(base_folder, filename):
    """
    Organise uploads into date-based subdirectories.
    /uploads/2024/01/15/a3f8b2c1.jpg
    """
    today = datetime.utcnow()
    date_path = os.path.join(
        base_folder,
        str(today.year),
        f"{today.month:02d}",
        f"{today.day:02d}",
    )
    os.makedirs(date_path, exist_ok=True)
    return os.path.join(date_path, filename)

def get_relative_path(base_folder, full_path):
    """Get the relative path for storing in the database."""
    return os.path.relpath(full_path, base_folder)

base = '/var/uploads'
test_files = ['a3f8b2c1.jpg', 'd4e5f6a7.png']

print("Date-based directory structure:")
for f in test_files:
    full = get_upload_path(base, f)
    rel = get_relative_path(base, full)
    print(f"  Full path: {full}")
    print(f"  In DB:     {rel}")
    print()

print("Benefits:")
print("  - No single directory with millions of files")
print("  - Easy to archive old uploads (delete /uploads/2022/)")
print("  - Natural partitioning for backups")
print("  - Meaningful CDN/S3 key structure")

### Handling Multiple File Uploads

In [ ]:
from flask import Flask, request, jsonify
from werkzeug.utils import secure_filename
import os, uuid

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = '/tmp/uploads'
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024  # 50MB total
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif'}
MAX_FILES = 10

@app.route('/upload/multiple', methods=['POST'])
def upload_multiple():
    """Handle <input type='file' multiple>."""
    files = request.files.getlist('photos')  # returns a list of FileStorage

    if not files:
        return jsonify(error='No files provided'), 400
    if len(files) > MAX_FILES:
        return jsonify(error=f'Maximum {MAX_FILES} files allowed'), 400

    results, errors = [], []
    for file in files:
        if file.filename == '':
            continue
        ext = file.filename.rsplit('.', 1)[-1].lower() if '.' in file.filename else ''
        if ext not in ALLOWED_EXTENSIONS:
            errors.append({"file": file.filename, "error": "Extension not allowed"})
            continue
        filename = uuid.uuid4().hex + '.' + ext
        os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
        file.save(os.path.join(app.config['UPLOAD_FOLDER'], filename))
        results.append({"original": file.filename, "saved_as": filename})

    return jsonify(uploaded=results, errors=errors, total=len(results)), 201 if results else 400

print("Multiple file uploads:")
print("  request.files.getlist('photos') -> list of FileStorage objects")
print("  HTML: <input type='file' name='photos' multiple accept='image/*'>")

### Serving Uploaded Files Safely with `send_from_directory()`

In [ ]:
from flask import Flask, send_from_directory, abort
import os

app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = '/tmp/uploads'

# SAFE: send_from_directory prevents path traversal
@app.route('/uploads/<path:filename>')
def serve_upload(filename):
    """
    send_from_directory() protects against path traversal.
    You cannot escape the configured UPLOAD_FOLDER.
    """
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

# With access control
@app.route('/secure-uploads/<path:filename>')
def serve_secure(filename):
    """Only serve files to authorised users."""
    # file_record = FileRecord.query.filter_by(filename=filename).first_or_404()
    # if file_record.user_id != current_user.id:
    #     abort(403)
    return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

# UNSAFE (never do this):
# @app.route('/file/<path:filename>')
# def unsafe(filename):
#     with open('/uploads/' + filename, 'rb') as f:  # Path traversal!
#         return f.read()

print("send_from_directory() protection:")
print()
print("  URL: /uploads/../../etc/passwd")
print("  Werkzeug detects the traversal -> 403 Forbidden")
print("  (Not 200 with the password file!)")
print()
print("Production tip: Let nginx or S3 serve files directly.")
print("Flask serving uploads creates unnecessary application server load.")

### Flask's Static File System

Built-in support for CSS, JS, and app images (different from user uploads):

In [ ]:
# Flask's static/ directory structure:
#
# myapp/
# ├── static/               <- Flask auto-serves at /static/
# |   ├── css/style.css      <- /static/css/style.css
# |   ├── js/app.js          <- /static/js/app.js
# |   └── img/logo.png       <- /static/img/logo.png
# └── templates/

html_examples = """
<!-- CORRECT: url_for() generates the right URL even at sub-paths -->
<link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
<script src="{{ url_for('static', filename='js/app.js') }}"></script>
<img src="{{ url_for('static', filename='img/logo.png') }}">

<!-- WRONG: hardcoded path breaks if app is mounted at a sub-path -->
<link rel="stylesheet" href="/static/css/style.css">
"""

print("Flask static file serving:")
print()
print("  app.static_folder   = 'static'   (default)")
print("  app.static_url_path = '/static'  (default)")
print()
print("Custom static folder:")
print("  app = Flask(__name__, static_folder='assets', static_url_path='/assets')")
print("  url_for('static', filename='style.css') -> /assets/style.css")
print()
print("Cache control for production:")
print("  app.config['SEND_FILE_MAX_AGE_DEFAULT'] = 31536000  # 1 year")
print("  Adds: Cache-Control: max-age=31536000")

### Handling Upload Errors Gracefully

In [ ]:
from flask import Flask, request, jsonify
from werkzeug.exceptions import RequestEntityTooLarge

app = Flask(__name__)
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB

@app.errorhandler(RequestEntityTooLarge)
def file_too_large(e):
    """
    Flask raises 413 when Content-Length exceeds MAX_CONTENT_LENGTH.
    This happens BEFORE the view function is called.
    """
    max_mb = app.config['MAX_CONTENT_LENGTH'] // (1024 * 1024)
    return jsonify(error='File too large', max_size_mb=max_mb), 413

@app.route('/upload', methods=['POST'])
def upload():
    try:
        file = request.files.get('photo')
        if not file or file.filename == '':
            return jsonify(error='No file provided'), 400
        # ... process file
        return jsonify(message='Upload successful'), 201
    except OSError as e:
        app.logger.error(f'File save error: {e}')
        return jsonify(error='Could not save file'), 500

print("Upload error codes:")
print("  413 RequestEntityTooLarge  File exceeds MAX_CONTENT_LENGTH")
print("  400 Bad Request            No file / empty filename")
print("  415 Unsupported Media      Wrong file type")
print("  500 Server Error           Disk full, permissions, etc.")
print()
print("Flask checks Content-Length BEFORE reading the body.")
print("Oversized requests are rejected before your view runs.")

### ⚖️ Local Storage vs Cloud Storage

In [ ]:
print("Local Storage (filesystem)")
print("=" * 50)
for item in [
    "+ Simple: just file.save()",
    "+ Fast: no network latency",
    "+ No external dependencies or cost",
    "- Ephemeral on PaaS (Heroku/Fly wipes on restart)",
    "- Not shared across multiple instances (scaling breaks)",
    "- No CDN: slow for distant users",
    "- You manage backups, redundancy, storage limits",
    "- Serving through Python wastes application resources",
]:
    print(f"  {item}")

print()
print("Cloud Storage (AWS S3, GCS, Cloudinary)")
print("=" * 50)
for item in [
    "+ Persistent: survives server restarts",
    "+ Scales infinitely (no disk management)",
    "+ Global CDN (fast for all users)",
    "+ Built-in redundancy and backups",
    "+ Serves files directly (no Flask overhead)",
    "- External dependency to configure",
    "- Network latency on upload",
    "- Costs money (cheap, but not free)",
    "- More code to write",
]:
    print(f"  {item}")

print()
print("Rule of thumb:")
print("  Development / hobby projects -> local storage is fine")
print("  Production / multi-server    -> use cloud storage")

### Cloud Storage: AWS S3 Pattern

In [ ]:
# pip install boto3

print("AWS S3 upload pattern:")
print()
print("""
import boto3, uuid, os
from flask import Flask, request, jsonify

app = Flask(__name__)
s3 = boto3.client(
    's3',
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
    region_name='us-east-1'
)
BUCKET = os.environ['S3_BUCKET']
CDN_URL = f"https://{BUCKET}.s3.amazonaws.com"

@app.route('/upload', methods=['POST'])
def upload():
    file = request.files.get('photo')
    if not file or file.filename == '':
        return jsonify(error='No file'), 400

    ext = file.filename.rsplit('.', 1)[-1].lower()
    key = f"uploads/{uuid.uuid4().hex}.{ext}"  # S3 key

    # Upload directly to S3 (never touches your server's disk)
    s3.upload_fileobj(
        file.stream,
        BUCKET,
        key,
        ExtraArgs={'ContentType': file.content_type, 'ACL': 'public-read'},
    )

    file_url = f"{CDN_URL}/{key}"
    # post.cover_image_url = file_url
    # db.session.commit()
    return jsonify(url=file_url), 201
""")
print()
print("Cloudinary (even simpler + image transformations):")
print("  import cloudinary.uploader")
print("  result = cloudinary.uploader.upload(file.stream)")
print("  url = result['secure_url']")

## 🧪 What If?

### What if you skip `secure_filename()`?

In [ ]:
import os
from werkzeug.utils import secure_filename

malicious = "../../etc/passwd"
upload_folder = "/var/uploads"

# Without secure_filename:
unsafe_path = os.path.join(upload_folder, malicious)
print("WITHOUT secure_filename():")
print(f"  Input:          {malicious!r}")
print(f"  os.path.join:   {unsafe_path!r}")
print(f"  Normalized:     {os.path.normpath(unsafe_path)!r}")
print("  -> Would read/write: /etc/passwd  <- ATTACK SUCCEEDS")
print()

# With secure_filename:
safe_name = secure_filename(malicious)
print("WITH secure_filename():")
print(f"  Input:  {malicious!r}")
print(f"  Result: {safe_name!r}")
if safe_name:
    print(f"  Path:   {os.path.join(upload_folder, safe_name)!r}")
    print("  -> Saves to /var/uploads/etc_passwd (harmless)")
else:
    print("  -> Empty string! Check and reject.")
    print("  -> Attack completely neutralised.")

print()
print("Defence in depth alongside secure_filename():")
print("  - Mount UPLOAD_FOLDER on a separate volume")
print("  - Never put UPLOAD_FOLDER in Python module path")
print("  - Configure web server to NOT execute scripts in UPLOAD_FOLDER")

In [ ]:
# What if MAX_CONTENT_LENGTH is not set?
print("Without MAX_CONTENT_LENGTH:")
print()
print("  Attacker uploads a 10GB file:")
print("  python -c \"print('A' * 10**10)\" | curl -X POST \\")
print("           -F 'photo=@/dev/stdin' http://yourapp.com/upload")
print()
print("  -> Flask reads the entire stream before calling your view")
print("  -> Disk fills up -> other users get 500 errors")
print("  -> Potential server crash")
print()
print("With MAX_CONTENT_LENGTH = 16 * 1024 * 1024:")
print("  -> Flask reads Content-Length header immediately")
print("  -> If Content-Length > 16MB -> 413 RequestEntityTooLarge")
print("  -> Request body never read, upload rejected instantly")
print()
print("Defense in depth:")
print("  nginx: client_max_body_size 16M;  # web server level check")
print("  Flask: MAX_CONTENT_LENGTH        # application level check")

In [ ]:
# What if two users upload the same filename simultaneously?
print("Race condition without UUID filenames:")
print()
print("  User A uploads 'profile.jpg' at 10:00:00.001")
print("  User B uploads 'profile.jpg' at 10:00:00.002")
print()
print("  User A: file.save('/uploads/profile.jpg')  -> saved")
print("  User B: file.save('/uploads/profile.jpg')  -> OVERWRITES User A!")
print()
print("  Database still says User A has /uploads/profile.jpg")
print("  But the file now contains User B's photo!")
print()
print("With UUID filenames:")
print("  User A -> /uploads/a3f8b2c1d4e5f6a7.jpg  (unique)")
print("  User B -> /uploads/9b8c7d6e5f4a3b2c.jpg  (unique)")
print()
print("UUID v4 collision probability:")
print("  1 in 2^128 (approximately 10^38)")
print("  You'd need to generate 10^18 UUIDs/s for 85 years to have a 50%")
print("  chance of a single collision. Practically impossible.")

## 🚀 Real-World Project Link

**Our Blog's Cover Image Upload**

The Blog app uses file upload for post cover images — validated, renamed with UUID, stored in dated directories, and served via Flask:

```python
# blog/uploads.py
import os, uuid, imghdr
from datetime import datetime
from flask import current_app
from werkzeug.utils import secure_filename

ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg', 'gif', 'webp'}

def save_cover_image(file):
    """
    Validate and save a cover image.
    Returns relative path to store in the database.
    Raises ValueError on validation failure.
    """
    if not file or file.filename == '':
        raise ValueError('No file provided')

    ext = file.filename.rsplit('.', 1)[-1].lower() if '.' in file.filename else ''
    if ext not in ALLOWED_EXTENSIONS:
        raise ValueError(f'File type .{ext} not allowed')

    file.stream.seek(0)
    img_type = imghdr.what(file.stream)
    file.stream.seek(0)
    if not img_type:
        raise ValueError('File does not appear to be a valid image')

    today = datetime.utcnow()
    upload_dir = os.path.join(
        current_app.config['UPLOAD_FOLDER'],
        str(today.year), f"{today.month:02d}", f"{today.day:02d}"
    )
    os.makedirs(upload_dir, exist_ok=True)

    filename = uuid.uuid4().hex + '.' + ext
    file.save(os.path.join(upload_dir, filename))

    return os.path.relpath(
        os.path.join(upload_dir, filename),
        current_app.config['UPLOAD_FOLDER']
    )

# blog/routes/posts.py
# @posts.route('/new', methods=['POST'])
# @login_required
# def new_post():
#     try:
#         image_path = save_cover_image(request.files.get('cover_image'))
#     except ValueError as e:
#         flash(str(e), 'error')
#         return render_template('new_post.html', form=form)
#     post = Post(title=form.title.data, cover_image=image_path)
#     db.session.add(post)
#     db.session.commit()
```

## 📋 Chapter Summary & Cheat Sheet

### Key Takeaways

1. **Always use `secure_filename()`** — prevents path traversal. No exceptions.
2. **Three validation layers**: extension whitelist + MIME/magic bytes + file size limit
3. **Generate UUID filenames** — prevent collisions and information leakage
4. **`MAX_CONTENT_LENGTH`** — set it or face disk-exhaustion DoS attacks
5. **`send_from_directory()`** — the safe way to serve uploaded files
6. **Local storage is development-only** — use S3/Cloudinary in production
7. **Never trust `Content-Type` header** — read the magic bytes instead

### Cheat Sheet

```python
# Config
app.config['UPLOAD_FOLDER'] = '/var/uploads'
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB

# Essential imports
from werkzeug.utils import secure_filename
from flask import send_from_directory
import uuid, os, imghdr

# Validate extension
ALLOWED = {'png', 'jpg', 'jpeg', 'gif'}
ext = filename.rsplit('.', 1)[-1].lower()
if ext not in ALLOWED: return error, 415

# Validate MIME (magic bytes)
file.stream.seek(0)
img_type = imghdr.what(file.stream)  # None if not an image
file.stream.seek(0)

# Generate safe unique filename
filename = uuid.uuid4().hex + '.' + ext

# Save
file.save(os.path.join(app.config['UPLOAD_FOLDER'], filename))

# Serve
return send_from_directory(app.config['UPLOAD_FOLDER'], filename)

# Static files in templates
{{ url_for('static', filename='css/style.css') }}

# Handle 413 Too Large
@app.errorhandler(413)
def too_large(e):
    return jsonify(error='File too large'), 413
```

## 💪 Practice Prompts

1. **Profile Picture Upload**: Build a `/profile/photo` endpoint. Validate: max 2MB, PNG/JPEG only, check magic bytes. When a user uploads a new photo, delete their old one from disk (look it up from the database first). Store the new UUID filename. Redirect to the profile page on success.

2. **Drag-and-Drop Upload with Preview**: Create a page with a JavaScript drag-and-drop zone that uploads via `fetch()` to a Flask endpoint returning JSON. JavaScript shows a preview with `URL.createObjectURL()` before the upload, and then the server URL after. Display errors (wrong type, too large) without a page reload.

3. **Storage Backend Abstraction**: Create a `storage.py` module with `save_file(stream, filename)` and `get_file_url(filename)` functions. Implement `LocalStorage` (wraps `file.save()`) and `S3Storage` (uses boto3). Select backend via `app.config['STORAGE_BACKEND'] = 'local'` or `'s3'`. The upload route should not change — only the storage module swaps.

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../5.%20building_apis/13.%20api_authentication_with_jwt.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>← Chapter 13: JWT Auth</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>📚 Table of Contents</a>
  <a href='15.%20error_handling_and_logging.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 15: Error Handling →</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='../5. building_apis/13. api_authentication_with_jwt.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='15. error_handling_and_logging.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>